# Preprocessing

This notebook will hold the preprocessing steps for the Liberty Heritage dataset, including cleaning, encoding, and feature preparation.

## Planned Steps

- Load the raw dataset
- Review the columns selected for modeling
- Handle missing values and encode categorical features
- Prepare the final modeling table

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## Preprocessing Plan

Based on the data quality checks and EDA, the preprocessing plan is:

1. **Keep the raw audit intact.** Do not modify the source CSV in place; work on a copy inside the notebook or pipeline.
2. **Drop identifier-only fields from modeling.** `ApplicationID` should be excluded from features because it is unique and not predictive.
3. **Treat missing values explicitly.** The quality review showed missingness in `EmploymentLengthYears`, `AnnualIncome`, `YearsAtCurrentResidence`, `RevolvingUtilization`, and `DebtToIncomeRatio`. These should be handled column by column rather than with a blanket rule.
4. **Preserve binary indicators as binary features.** Columns such as `Employed`, `IncomeVerified`, `PriorDefault`, and `BankruptcyLast7Years` are stored as integers but function like flags.
5. **Encode categorical variables.** One-hot encode nominal fields such as `Gender`, `MaritalStatus`, `EducationLevel`, `State`, `ResidenceType`, `EmploymentType`, and any other non-ordinal categorical columns.
6. **Handle the target separately.** `Approval` should remain excluded from feature preprocessing and be encoded only at the modeling stage if required.
7. **Consider whether `fico_band` should be retained.** The EDA showed a strong approval gradient across standard FICO bands, so a derived banded feature may be useful for interpretation. If retained, it should be created from `CreditScore` only in the modeling pipeline, not written back to the raw data.
8. **Review highly correlated numeric features.** `Age` and `CreditHistoryMonths` are very strongly correlated, and there are additional moderate relationships among income, debt, and account-count variables. We should avoid unnecessary redundancy if model simplicity matters, but keep correlated features if the downstream model can use them effectively.
9. **Check skewed numeric variables.** Variables such as `AnnualIncome`, `TotalMonthlyDebtPayment`, `DebtToIncomeRatio`, and `RevolvingUtilization` may benefit from robust scaling or transformation depending on the final model choice.
10. **Build the final preprocessing pipeline reproducibly.** Any imputers, encoders, and transformations should be fit only on the training split and then applied consistently to validation or test data.

Open questions to resolve before implementation:

- Should missing numeric values be imputed with median, model-based estimates, or a separate missing indicator?
- Should `fico_band` be used as a derived feature, or should we keep only the raw `CreditScore`?
- Do we want to collapse any sparse categories, especially in `State` and `EmploymentType`, before encoding?

## Imputation Impact

The five numeric columns below are the ones with missing values in the raw dataset. If we use median imputation, these are the exact replacement values and the number of rows that would change.

In [2]:
quality_impact = pd.DataFrame(
    [
        {'column': 'EmploymentLengthYears', 'median': 2.0, 'rows_affected': 146},
        {'column': 'AnnualIncome', 'median': 62.4, 'rows_affected': 139},
        {'column': 'YearsAtCurrentResidence', 'median': 4.0, 'rows_affected': 96},
        {'column': 'RevolvingUtilization', 'median': 0.316, 'rows_affected': 85},
        {'column': 'DebtToIncomeRatio', 'median': 0.242, 'rows_affected': 75},
    ]
)

quality_impact['imputation_method'] = 'median'
quality_impact['rows_affected_pct'] = (quality_impact['rows_affected'] / 8000 * 100).round(2)

print(quality_impact.to_string(index=False))
print('\nNote: no cap/clip operation is planned yet, so only these missing-value replacements would change the data at this stage.')

                 column  median  rows_affected imputation_method  rows_affected_pct
  EmploymentLengthYears   2.000            146            median               1.82
           AnnualIncome  62.400            139            median               1.74
YearsAtCurrentResidence   4.000             96            median               1.20
   RevolvingUtilization   0.316             85            median               1.06
      DebtToIncomeRatio   0.242             75            median               0.94

Note: no cap/clip operation is planned yet, so only these missing-value replacements would change the data at this stage.


In [3]:
from pathlib import Path

raw_path = Path('data/raw/Liberty_Heritage_Data.csv')
if not raw_path.exists():
    raw_path = Path('../data/raw/Liberty_Heritage_Data.csv')

df = pd.read_csv(raw_path)

impute_medians = {
    'EmploymentLengthYears': 2.0,
    'AnnualIncome': 62.4,
    'YearsAtCurrentResidence': 4.0,
    'RevolvingUtilization': 0.316,
    'DebtToIncomeRatio': 0.242,
}

for column, median_value in impute_medians.items():
    df[column] = df[column].fillna(median_value)

cap_upper = {
    'AnnualIncome': 147.2,
    'DebtToIncomeRatio': 0.852,
    'RevolvingUtilization': 0.6833,
}

for column, upper_bound in cap_upper.items():
    df[column] = df[column].clip(upper=upper_bound)

out_path = Path('data/interim/cleaned.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)

print(f'Saved cleaned data to {out_path}')
print(f'Final shape: {df.shape}')
print(f'Final missing values: {int(df.isna().sum().sum())}')
print('New min/max after cleaning:')
print(f"DebtToIncomeRatio: min={df['DebtToIncomeRatio'].min():.4f}, max={df['DebtToIncomeRatio'].max():.4f}")
print(f"RevolvingUtilization: min={df['RevolvingUtilization'].min():.4f}, max={df['RevolvingUtilization'].max():.4f}")
print(f"AnnualIncome: min={df['AnnualIncome'].min():.4f}, max={df['AnnualIncome'].max():.4f}")


Saved cleaned data to data/interim/cleaned.csv
Final shape: (8000, 25)
Final missing values: 0
New min/max after cleaning:
DebtToIncomeRatio: min=0.0200, max=0.8520
RevolvingUtilization: min=0.0000, max=0.6833
AnnualIncome: min=12.0000, max=147.2000


## Feature Engineering

The cleaned dataset is now extended with two binary flags: `ThinFile` and `HighUtilization`. The `HighUtilization` flag is evaluated after the prior capping step, so its count may be affected by that cleaning choice.

In [4]:
from pathlib import Path

processed = df.copy()
processed['ThinFile'] = (processed['CreditHistoryMonths'] < 24).astype(int)
processed['HighUtilization'] = (processed['RevolvingUtilization'] > 0.7).astype(int)

out_path = Path('data/processed/featured.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
processed.to_csv(out_path, index=False)

print(f'Saved featured data to {out_path}')
print(f'Final shape: {processed.shape}')
print(f"ThinFile 1s: {int(processed['ThinFile'].sum())}")
print(f"HighUtilization 1s: {int(processed['HighUtilization'].sum())}")


Saved featured data to data/processed/featured.csv
Final shape: (8000, 27)
ThinFile 1s: 440
HighUtilization 1s: 0
